In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
# 1. 数据读取与严格对齐
# 读取 train_val 与 test 两个分片的 metadata，并合并
meta_train_val = pd.read_csv('WAYB_WAYC_metadata_train_val(1).csv')
meta_test = pd.read_csv('WAYB_WAYC_metadata_test(1).csv')
meta = pd.concat([meta_train_val, meta_test], ignore_index=True)

# 读取 train_val 与 test 两个分片的蛋白丰度，并合并
protein_train_val = pd.read_csv('WAYB_WAYC_proteome_raw_train_val.csv')
protein_test = pd.read_csv('WAYB_WAYC_proteome_raw_test.csv')
protein = pd.concat([protein_train_val, protein_test], ignore_index=True)

print('meta shape:', meta.shape)
print('protein shape:', protein.shape)

# 将索引设置为 'sample_ID'
meta = meta.set_index('sample_ID')
protein = protein.set_index('sample_ID')

# 取交集，仅保留两边都存在的样本（Inner Join）
common_ids = meta.index.intersection(protein.index)
meta = meta.loc[common_ids]
protein = protein.loc[common_ids]

print(f"对齐后总样本数: {len(common_ids)}")

meta shape: (13412, 15)
protein shape: (13412, 5244)


对齐后总样本数: 13412


In [3]:
# 2. 特征/标签过滤（严格仅在训练集上计算，防止数据泄漏）
train_mask = meta['split_final'] == 'train'
train_protein = protein.loc[train_mask]

# 计算训练集中各蛋白列的缺失率
missing_rate = train_protein.isna().mean()

# 筛选并仅保留缺失率 < 80% 的蛋白列
valid_protein_cols = missing_rate[missing_rate < 0.8].index
print(f"保留的蛋白列数量 (缺失率 < 80%): {len(valid_protein_cols)}")

# 仅保留筛选后的蛋白列
protein_filtered = protein[valid_protein_cols]

保留的蛋白列数量 (缺失率 < 80%): 4422


In [4]:
# 3. 数据转换与 Mask 保留
# 对筛选后的蛋白表达量矩阵执行 np.log2() 变换，NaN 保持为 NaN（缺失 Mask）
protein_log2 = np.log2(protein_filtered)

# 验证 NaN 仍然是 NaN（缺失 Mask 保留）
print(f"原始 NaN 数量: {protein_filtered.isna().sum().sum()}")
print(f"log2 后 NaN 数量: {protein_log2.isna().sum().sum()}")
print(f"NaN 保持一致: {protein_filtered.isna().sum().sum() == protein_log2.isna().sum().sum()}")

原始 NaN 数量: 8974734
log2 后 NaN 数量: 8974734


NaN 保持一致: True


In [5]:
# 4. Baseline 1 均值预测生成
# 训练集的 log2 蛋白矩阵
train_log2 = protein_log2.loc[train_mask]

# 计算训练集中每个蛋白列在非缺失值下的 log2 平均值，得到全局蛋白均值向量
protein_mean_vector = train_log2.mean(axis=0, skipna=True)

# 验证集 (split_final 以 'val' 开头) 的所有样本
val_mask = meta['split_final'].str.startswith('val')
val_log2 = protein_log2.loc[val_mask]

# 构建预测矩阵：每一个验证集样本的所有蛋白预测值都等于该 protein_mean_vector
val_preds = pd.DataFrame(
    np.tile(protein_mean_vector.values, (val_log2.shape[0], 1)),
    index=val_log2.index,
    columns=val_log2.columns
)

print(f"验证集样本数: {val_log2.shape[0]}")
print(f"验证集预测矩阵 shape: {val_preds.shape}")

验证集样本数: 3038
验证集预测矩阵 shape: (3038, 4422)


In [6]:
# 5. 评估指标计算与打印
val_true = val_log2  # 验证集真实的 log2 蛋白表达量矩阵

# 在计算评估指标时，只针对真实值 val_true 中非 NaN（非缺失）的位置进行评估
valid_mask = val_true.notna().values  # numpy 布尔数组

# 提取有效位置的真实值和预测值（1D 数组）
y_true_valid = val_true.values[valid_mask]
y_pred_valid = val_preds.values[valid_mask]

# 计算总体 RMSE (Root Mean Squared Error)
rmse = np.sqrt(np.mean((y_true_valid - y_pred_valid) ** 2))

# 计算全局 R2 Score（只传入非缺失有效值对）
r2 = r2_score(y_true_valid, y_pred_valid)

print(f"===== Baseline 1 (全局蛋白均值基线) 结果 =====")
print(f"对齐后总样本数: {len(common_ids)}")
print(f"保留的蛋白列数量: {len(valid_protein_cols)}")
print(f"验证集样本数: {val_log2.shape[0]}")
print(f"有效评估位置数 (非缺失): {valid_mask.sum()}")
print(f"RMSE: {rmse:.6f}")
print(f"R2 Score: {r2:.6f}")

===== Baseline 1 (全局蛋白均值基线) 结果 =====
对齐后总样本数: 13412
保留的蛋白列数量: 4422
验证集样本数: 3038
有效评估位置数 (非缺失): 11521736
RMSE: 0.938526
R2 Score: 0.885639
